# Phylogeographic Visualization of Lassa Virus Lineage IV

This notebook generates phylogeographic visualizations for LASV lineage IV sequences
from Guinea, Sierra Leone, and Liberia.

**Contents:**

1. Setup & Dependencies
2. Data Loading (Trees & Geographic Data)
3. Phylogeographic Maps (L & S Segments)
4. Zoomed Inset Maps
5. Supplementary Tree Visualizations with City Labels


In [ ]:
# ============================================================================
# SETUP & DEPENDENCIES
# ============================================================================

# Phylogeographic visualization
library(ggphylogeo)
library(ggtree)
library(phangorn)

# Data manipulation
library(dplyr)
library(stringr)
library(purrr)
library(tidyr)
library(readxl)

# Geographic data
library(rnaturalearth)
library(sf)
library(terra)

# Plotting
library(ggplot2)
library(ggrepel)
library(patchwork)

# Helper function
`%notin%` <- Negate(`%in%`)

In [ ]:
sessionInfo()

In [ ]:
new_names <- read_xlsx("../../data/new-guinea-data/GIN-LASV-final-ViMOP-seq-for-Leuven_V02.xlsx")%>%
    select(`seq-ID`, contains("Genebank"))%>%
    pivot_longer(cols = contains("Genebank"), names_to = "segment", values_to = "NCBID")%>%
    #ends with S or L
    mutate(
        segment = ifelse(grepl("S$", segment), "S", "L"),
        seqID= ifelse(grepl("S$", segment), paste0(`seq-ID`, "-S"), paste0(`seq-ID`, "-L")),
        sampleID= `seq-ID`)%>%
    select(seqID,sampleID, segment, NCBID)

# Lookup table for mapping seqID -> NCBID labels
NCBID_LOOKUP <- new_names %>%
    filter(!is.na(NCBID)) %>%
    distinct(seqID, NCBID) %>%
    {setNames(.$NCBID, .$seqID)}

# Helper to map seqID to NCBID (fallback to seqID if missing)
label_from_seqid <- function(seq_id) {
    mapped <- NCBID_LOOKUP[seq_id]
    ifelse(is.na(mapped) | mapped == "", seq_id, mapped)
}

## 1. Load Phylogenetic Trees

Load BEAST-annotated trees for L and S segments.


In [ ]:
# Load BEAST MCC trees
L_tree <- treeio::read.beast("./L.12.combined.ann.tree")
S_tree <- treeio::read.beast("./S.12.combined.ann.tree")

In [ ]:
autoplot.treedata(L_tree, most_recent_sample = "2024-09-19")

In [ ]:
autoplot.treedata(S_tree, most_recent_sample = "2024-09-19")

## 2. Build Phylogeographic Data Structures

Convert tree data to phylogeographic format for map visualization.


In [ ]:
# Build phylogeographic structures with most recent sample date
MRSD <- "2024-09-19"

L <- build_phylogeo(L_tree, most_recent_sample = MRSD)
S <- build_phylogeo(S_tree, most_recent_sample = MRSD)

## 3. Load Geographic Data

Load Natural Earth and custom geographic layers for the West African region.


In [ ]:
# Countries of interest in West Africa
COUNTRIES_INTEREST <- c(
    "Ivory Coast", "Guinea", "Guinea-Bissau", "Liberia",
    "Sierra Leone", "Mali", "Senegal", "Gambia",
    "Burkina Faso", "Ghana", "Togo", "Benin"
)

# Download Natural Earth data
countries <- ne_download(scale = 10, type = "countries", category = "cultural", returnclass = "sf")
wa_countries <- countries[countries$ADMIN %in% COUNTRIES_INTEREST, ]

states <- ne_download(scale = 10, type = "admin_1_states_provinces", category = "cultural", returnclass = "sf")
wa_states <- states[states$admin %in% COUNTRIES_INTEREST, ]

cities <- ne_download(scale = 10, type = "populated_places", category = "cultural", returnclass = "sf")

# Load custom shapefiles
borders <- st_read("Environmental_files/Natural_Earth/International_borders.shp")
lakes   <- st_read("Environmental_files/Natural_Earth/Natural_Earth_lakes.shp")
rivers  <- st_read("Environmental_files/LASV_rivers/FAO_GeoNetwork.shp")

In [ ]:
# Style rivers based on Strahler stream order
rivers <- rivers %>%
    mutate(
        Strahler    = ifelse(Strahler < 3, 0, Strahler),
        river_alpha = ifelse(Strahler < 3, 0.2, 0.7),
        river_size  = ifelse(Strahler < 3, 0.4, 0.6)
    )

In [ ]:
# Custom city labels with manual nudge values for map positioning
CITIES_OF_INTEREST <- data.frame(
    city    = c("Conakry", "Makeni", "Faranah", "Kissidougou", "Nzérékoré",
                "Gbarnga", "Kakata", "Kenema", "Bo", "Ganta", "Madina-Oula", "Guéckédou"),
    nudge_y = c(-0.25, -0.15, 0.4, 0.25, 0.25, -0.25, -0.25, -0.33, 0.15, -0.25, 0.15, 0.2),
    nudge_x = c(0,0,0,0,0,0,0,0,0,0,0,0.6)
)

# Update cities with custom coordinates where needed
cities_updated <- cities %>%
    full_join(CITIES_OF_INTEREST, by = c("NAME" = "city")) %>%
    mutate(
        # Add missing cities with manual coordinates
        LATITUDE  = case_when(
            # toupper(NAME) == "MADINA-OULA" ~ 9.878589,
            toupper(NAME) == "GANTA"       ~ 7.225,
            toupper(NAME) == "GUÉCKÉDOU"   ~ 8.562152,
            TRUE ~ LATITUDE
        ),
        LONGITUDE = case_when(
            # toupper(NAME) == "MADINA-OULA" ~ -12.446531,
            toupper(NAME) == "GANTA"       ~ -8.975,
            toupper(NAME) == "GUÉCKÉDOU"   ~ -10.133471,
            TRUE ~ LONGITUDE
        )
    ) %>%
    filter(!is.na(LATITUDE) & !is.na(LONGITUDE)) %>%
    st_drop_geometry() %>%
    st_as_sf(coords = c("LONGITUDE", "LATITUDE"), crs = st_crs(cities))

## 4. Create Base Map

Build the base map layer with geographic features (rivers, lakes, borders).


In [ ]:
# Base map with geographic features
mapplot <- ggplot() +
    # Country fill
    geom_sf(data = wa_countries, fill = "white") +
    # Lakes
    geom_sf(data = lakes, fill = "#AFBED5", color = "steelblue", alpha = 0.6) +
    # Rivers (styled by Strahler order)
    geom_sf(data = rivers, aes(size = river_size, alpha = river_alpha), color = "#AFBED5") +
    scale_size_identity() +
    scale_alpha_identity() +
    # Country borders
    geom_sf(data = borders, fill = "white", color = "gray30", size = 0.7)

# Preview base map with cities
mapplot +
    theme_minimal() +
    theme(panel.background = element_rect(fill = "gray90")) +
    geom_sf(data = cities_updated[cities_updated$NAME %in% CITIES_OF_INTEREST$city, ],
            color = "red", size = 2) +
    geom_sf_text(data = cities_updated[cities_updated$NAME %in% CITIES_OF_INTEREST$city, ],
                 aes(label = NAME, nudge_y = nudge_y, nudge_x = nudge_x), size = 4.5) +
    coord_sf(xlim = c(-15, -7), ylim = c(5, 12), expand = FALSE)

## 5. Define Sequences of Interest

Specify new sequences and those to highlight on the maps.


In [ ]:
# New sequences from this study
NEW_SEQUENCES <- c(
    "M00008", "M00009", "G0405", "G0274", "M00362", "M00363", "M00364",
    "M00539", "M00541", "M00542", "G0671", "G0683", "G0780", "G0795",
    "G0796", "G0797", "G0870", "G0900", "G0917", "G0920", "G0934",
    "G0953", "G0959", "G0960", "G0984", "G1013", "G1021", "G1029"
)

# Sequences to highlight with labels
HIGHLIGHT_SEQUENCES <- data.frame(
    label   = c("G0780", "G0795", "G0796", "G0797", "M00362", "M00542"),
    nudge_x = c(0, 0, 0, 0, -0.4, 0)
)

# Regex patterns for matching sequence IDs
REGEX_NEW <- paste0(".*(", paste(NEW_SEQUENCES, collapse = ")|("), ").*")
REGEX_HIGHLIGHT <- paste0(".*(", paste(HIGHLIGHT_SEQUENCES$label, collapse = ")|("), ").*")

## 6. Create Phylogeographic Maps

Generate phylogeographic visualizations for L and S segments.


In [ ]:
# Common viridis scale for year coloring
YEAR_SCALE <- scale_fill_viridis_c(
    name      = "Year",
    direction = -1,
    labels    = function(x) format(as.Date(x), "%Y"),
    breaks    = seq(as.Date("1700-01-01"), as.Date("2000-01-01"), by = "100 year"),
    limits    = as.numeric(as.Date(c("1650-01-01", "2050-01-01")))
)

# Common theme for phylogeographic maps
PHYLOGEO_THEME <- theme(
    axis.title.x   = element_blank(),
    axis.title.y   = element_blank(),
    axis.text      = element_text(size = 12),
    panel.border   = element_rect(color = "black", fill = NA, linewidth = 1.5)
)

# L Segment phylogeographic map
L_plot <- mapplot +
    geom_phylo_hpd(data = L, alpha = 0.2) +
    geom_phylo_branches(data = L) +
    geom_point(data = L$nodes %>% filter(!istip),aes(x = lon, y = lat, fill = age), shape = 22, size = 2) +
    # Background tips (existing sequences)
    geom_jitter(
        data = L$nodes %>% filter(istip & !grepl(REGEX_NEW, label)),
        inherit.aes = FALSE,
        aes(x = lon, y = lat, fill = age),
        shape = 21, size = 3
    ) +
    # New sequences (red)
    geom_jitter(
        data = L$nodes %>% filter(istip & grepl(REGEX_NEW, label)),
        inherit.aes = FALSE,
        aes(x = lon, y = lat),
        fill = "red", shape = 21, size = 3
    ) +
    # Highlighted sequence labels
    geom_text_repel(
        data = L$nodes %>%
            filter(istip & grepl(REGEX_HIGHLIGHT, label)) %>%
            mutate(
                sequence_id = str_extract(label, paste(HIGHLIGHT_SEQUENCES$label, collapse = "|")),
                seq_id_with_seg = paste0(sequence_id, "-L"),
                ncbid_label = label_from_seqid(seq_id_with_seg)
            ) %>%
            left_join(HIGHLIGHT_SEQUENCES, by = c("sequence_id" = "label")),
        aes(x = lon + nudge_x, y = lat, label = ncbid_label),
        size = 3, nudge_y = 0.2, label.size = 0.2, check_overlap = TRUE
    ) +
    YEAR_SCALE +
    geom_text(aes(x = -12.75, y = 5.75, label = "Segment L"), size = 7, fontface = "bold") +
    geom_sf_text(
        data = cities_updated[cities_updated$NAME %in% CITIES_OF_INTEREST$city, ],
        aes(label = NAME, nudge_y = nudge_y, nudge_x = nudge_x), size = 5
    ) +
    coord_sf(xlim = L$viewbox$xlim, ylim = L$viewbox$ylim) +
    theme_phylogeo() +
    guides_phylogeo() +
    scale_x_continuous(breaks = c(-15, -8), labels = c("15°W", "8°W")) +
    scale_y_continuous(breaks = c(6, 11), labels = c("6°N", "11°N")) +
    PHYLOGEO_THEME

L_plot

In [ ]:
# S Segment phylogeographic map
S_plot <- mapplot +
    geom_phylo_hpd(data = S, alpha = 0.2) +
    geom_phylo_branches(data = S) +
    geom_point(data = S$nodes %>% filter(!istip), aes(x = lon, y = lat, fill = age), size = 2, shape = 22) +
    # Background tips (existing sequences)
    geom_jitter(
        data = S$nodes %>% filter(istip & !grepl(REGEX_NEW, label)),
        inherit.aes = FALSE,
        aes(x = lon, y = lat, fill = age),
        shape = 21, size = 3
    ) +
    # New sequences (red)
    geom_jitter(
        data = S$nodes %>% filter(istip & grepl(REGEX_NEW, label)),
        inherit.aes = FALSE,
        aes(x = lon, y = lat),
        fill = "red", shape = 21, size = 3
    ) +
    # Highlighted sequence labels
    geom_text_repel(
        data = S$nodes %>%
            filter(istip & grepl(REGEX_HIGHLIGHT, label)) %>%
            mutate(
                sequence_id = str_extract(label, paste(HIGHLIGHT_SEQUENCES$label, collapse = "|")),
                seq_id_with_seg = paste0(sequence_id, "-S"),
                ncbid_label = label_from_seqid(seq_id_with_seg)
            ) %>%
            left_join(HIGHLIGHT_SEQUENCES, by = c("sequence_id" = "label")),
        aes(x = lon + nudge_x, y = lat, label = ncbid_label),
        size = 3, nudge_y = 0.2, label.size = 0.2, check_overlap = TRUE
    ) +
    YEAR_SCALE +
    geom_text(aes(x = -12.75, y = 5.75, label = "Segment S"), size = 7, fontface = "bold") +
    geom_sf_text(
        data = cities_updated[cities_updated$NAME %in% CITIES_OF_INTEREST$city, ],
        aes(label = NAME, nudge_y = nudge_y, nudge_x = nudge_x), size = 5
    ) +
    coord_sf(xlim = L$viewbox$xlim, ylim = L$viewbox$ylim) +
    theme_phylogeo() +
    guides_phylogeo() +
    scale_x_continuous(breaks = c(-15, -8), labels = c("15°W", "8°W")) +
    scale_y_continuous(breaks = c(6, 11), labels = c("6°N", "11°N")) +
    PHYLOGEO_THEME

S_plot

In [ ]:
# Combined L and S segment maps
options(repr.plot.width = 17, repr.plot.height = 6)

combined_plot <- L_plot + S_plot + plot_layout(ncol = 2, nrow = 1)
combined_plot

# ggsave(
#     "./figures/phylogeo_LS_combined.pdf",
#     combined_plot,
#     width = 17, height = 6,
#     device = cairo_pdf,
#     create.dir = TRUE
# )

## 7. Zoomed Inset Maps

Create zoomed views for Kissidougou and Nzérékoré regions.


In [ ]:
# Define zoom regions for inset maps
ZOOM_BOXES <- list(
    Kissidougou = list(xlim = c(-11, -9.5), ylim = c(8, 9.5)),
    Nzerekore   = list(xlim = c(-9.7, -8.3), ylim = c(6.8, 8.3)),
    Faranah    = list(xlim = c(-11.2, -10.5), ylim = c(9.5, 10))
)

In [ ]:
#' Filter nodes to a geographic bounding box
#' @param nodes Data frame with lon/lat columns
#' @param box List with xlim and ylim vectors
filter_nodes_to_box <- function(nodes, box) {
    nodes %>%
        filter(
            lon >= box$xlim[1], lon <= box$xlim[2],
            lat >= box$ylim[1], lat <= box$ylim[2]
        )
}

#' Create a zoomed phylogeographic map for a specific region
#' @param S Phylogeographic object
#' @param mapplot Base ggplot map
#' @param box Bounding box (list with xlim/ylim)
#' @param regex Regex pattern for target sequences
#' @param title Plot title
make_zoom_plot <- function(S, mapplot, box, regex, title = NULL) {

    # Filter nodes to bounding box
    nodes_zoom <- filter_nodes_to_box(S$nodes, box)

    # Pre-calculate jittered positions (consistent across layers)
    set.seed(123)
    nodes_plot_data <- nodes_zoom %>%
        mutate(
            lon_final   = ifelse(istip, jitter(lon), lon),
            lat_final   = ifelse(istip, jitter(lat), lat),
            is_target   = istip & grepl(regex, label),
            seq_id      = str_split(label, "\\|", simplify = TRUE)[, 2],
            clean_label = label_from_seqid(seq_id)
        )

    mapplot +
        geom_phylo_hpd(data = S, alpha = 0.25) +
        geom_phylo_branches(data = S) +
        # Internal nodes
        geom_point(
            data = nodes_plot_data %>% filter(!istip),
            aes(x = lon_final, y = lat_final, fill = age),
            shape = 22,
            size = 2.5
        ) +
        # Background tips
        geom_point(
            data = nodes_plot_data %>% filter(istip & !is_target),
            inherit.aes = FALSE,
            aes(x = lon_final, y = lat_final, fill = age),
            shape = 21, size = 3
        ) +
        # Target tips (new sequences)
        geom_point(
            data = nodes_plot_data %>% filter(is_target),
            inherit.aes = FALSE,
            aes(x = lon_final, y = lat_final),
            fill = "red", shape = 21, size = 3
        ) +
        # Labels (using all points as repel obstacles)
        geom_text_repel(
            data = nodes_plot_data,
            aes(x = lon_final, y = lat_final, label = ifelse(is_target, clean_label, "")),
            size = 4, box.padding = 0.4, label.padding = 0.2,
            point.padding = 0.3, force_pull = 4, max.overlaps = 18
        ) +
        coord_sf(xlim = box$xlim, ylim = box$ylim, expand = FALSE, clip = "on") +
        scale_x_continuous(breaks = box$xlim) +
        scale_y_continuous(breaks = box$ylim) +
        YEAR_SCALE +
        geom_sf_text(
            data = cities_updated[cities_updated$NAME %in% CITIES_OF_INTEREST$city, ],
            aes(label = NAME, nudge_y = nudge_y * 0.8, nudge_x = nudge_x * 0.5),
            fontface = "bold", size = 5
        ) +
        theme_phylogeo() +
        theme(
            legend.position = "none",
            panel.border    = element_rect(color = "black", fill = NA, linewidth = 1.2),
            plot.title      = element_text(face = "bold", size = 15, hjust = 0.5),
            axis.title      = element_blank(),
            axis.text       = element_text(size = 12)
        ) +
        labs(title = title)
}

# Create zoomed plots
S_kissidougou_plot <- make_zoom_plot(S, mapplot, ZOOM_BOXES$Kissidougou, REGEX_NEW, "Guéckédou (Segment S)")
S_nzerekore_plot   <- make_zoom_plot(S, mapplot, ZOOM_BOXES$Nzerekore, REGEX_NEW, "Nzérékoré (Segment S)")

In [ ]:
make_zoom_plot(S, mapplot, list(xlim = c(-11.2, -10.5), ylim = c(9.75, 10.25)), REGEX_NEW, "Faranah (Segment S)")

In [ ]:
# Dispersion direction legend
dispersion_legend <- ggplot() +
    geom_curve(
        data = data.frame(
            x = c(1, 2), y = c(1.1, 1),
            xend = c(2, 1), yend = c(1.1, 1)
        ),
        aes(x = x, y = y, xend = xend, yend = yend),
        curvature = -0.3,
        arrow = arrow(length = unit(0.08, "inches"), type = "closed", ends = "first"),
        colour = "gray30", linewidth = 0.6
    ) +
    geom_text(aes(x = 1.5, y = 1.5, label = "Dispersion direction"), size = 7, fontface = "bold") +
    coord_cartesian(xlim = c(0.5, 2.5), ylim = c(0.6, 1.7), expand = FALSE) +
    theme_minimal() +
    theme(
        axis.title = element_blank(), axis.text = element_blank(), axis.ticks = element_blank(),
        panel.grid = element_blank(),
        panel.border = element_rect(colour = "black", fill = NA, linewidth = 1.2),
        plot.margin = margin(6, 6, 6, 6)
    )

In [ ]:
#' Convert bounding box to sf geometry for overlay
zoom_to_sf <- function(box) {
    st_as_sfc(st_bbox(
        c(xmin = box$xlim[1], xmax = box$xlim[2], ymin = box$ylim[1], ymax = box$ylim[2]),
        crs = 4326
    ))
}

# S plot with zoom box indicators
S_plot_with_boxes <- S_plot +
    geom_sf(data = zoom_to_sf(ZOOM_BOXES$Kissidougou), fill = NA, color = "black", linewidth = 1, linetype = "dashed") +
    geom_sf(data = zoom_to_sf(ZOOM_BOXES$Nzerekore), fill = NA, color = "black", linewidth = 1, linetype = "dashed") +
    coord_sf(xlim = L$viewbox$xlim, ylim = L$viewbox$ylim)

In [ ]:
# Composite figure layout:
# AAAABBBB  (A = L segment, B = S segment with boxes)
# AAAABBBB
# ##CCCDDD  (C = Kissidougou zoom, D = Nzérékoré zoom)
# EECCCDDD  (E = dispersion legend)

LAYOUT <- '
AAAABBBB
AAAABBBB
AAAABBBB
AAAABBBB
##CCCDDD
EECCCDDD
##CCCDDD
'

options(repr.plot.width = 15, repr.plot.height = 12)

final_plot <- wrap_plots(
    A = L_plot,
    B = S_plot_with_boxes,
    C = S_kissidougou_plot,
    D = S_nzerekore_plot,
    E = dispersion_legend,
    design = LAYOUT
) +
    plot_annotation(
        tag_levels = list(c("A", "B", "C", "D", "")),
    ) &
    theme(
        plot.tag = element_text(size = 18, face = "bold"), 
        plot.margin = margin(t = 2, r = 10, b = 2, l = 2)
    )

final_plot

ggsave("./figures/phylogeo_LS_with_S_zooms.pdf", final_plot,
       width = 15, height = 12, device = cairo_pdf)

---

## 8. Supplementary: Tree Visualizations with City Labels

Time-scaled phylogenetic trees with geographic annotation by nearest city.


In [ ]:
#' Find the nearest city for each node based on coordinates
#' @param metadata_df Data frame with location1_median (lat) and location2_median (lon)
#' @param cities_sf sf object with city points
#' @return Character vector of nearest city names
find_nearest_city <- function(metadata_df, cities_sf) {
    cities_coords <- st_coordinates(cities_sf)
    cities_lat <- cities_coords[, "Y"]
    cities_lon <- cities_coords[, "X"]

    nearest_cities <- character(nrow(metadata_df))

    for (i in seq_len(nrow(metadata_df))) {
        lat <- metadata_df$location1_median[i]
        lon <- metadata_df$location2_median[i]

        if (is.na(lat) | is.na(lon)) {
            nearest_cities[i] <- NA_character_
        } else {
            distances <- sqrt((lat - cities_lat)^2 + (lon - cities_lon)^2)
            nearest_cities[i] <- cities_sf$NAME[which.min(distances)]
        }
    }

    return(nearest_cities)
}

In [ ]:
# Load metadata for new sequences
new_samples <- read.csv("./new-seqs.tsv", sep = "\t") %>%
    mutate(newsequence = newsequence == "true")

# Country color palette
COUNTRY_COLORS <- c(
    'GIN' = '#E41A1C', 'SLE' = '#377EB8', 'LBR' = '#4DAF4A',
    'MLI' = '#984EA3', 'NGA' = '#FF7F00', 'CIV' = '#FFFF33',
    'SEN' = '#A65628', 'GHA' = '#F781BF'
)

# Cities to use for labeling
cities_for_labeling <- cities_updated %>% filter(NAME %in% CITIES_OF_INTEREST$city)

# --- S Segment Tree ---
S_tree <- treeio::read.beast("./S.12.combined.ann.tree")
S_metadata <- read.csv("../phylogeographic/runs/S/LASV_GENBANK_20250725_curated.S.noNIG-CIV.beast2.metadata_S.csv") %>%
    left_join(new_samples, by = c("segment_S" = "sequence_name"))

S_tree@phylo <- midpoint(S_tree@phylo)
S_tree_metadata <- left_join(S_tree, S_metadata, by = c("label" = "segment_S"))
S_tree_metadata@data$nearest_city <- find_nearest_city(S_tree_metadata@data, cities_for_labeling)

# --- L Segment Tree ---
L_tree <- treeio::read.beast("./L.12.combined.ann.tree")
L_metadata <- read.csv("../phylogeographic/runs/L/LASV_GENBANK_20250725_curated.L.noNIG-CIV.beast2.metadata_L.csv") %>%
    left_join(new_samples, by = c("segment_L" = "sequence_name"))

L_tree@phylo <- midpoint(L_tree@phylo)
L_tree_metadata <- left_join(L_tree, L_metadata, by = c("label" = "segment_L"))
L_tree_metadata@data$nearest_city <- find_nearest_city(L_tree_metadata@data, cities_for_labeling)

### 8.1 S Segment Tree with City Labels

Helper visualization to identify node numbers for clade configuration.


In [ ]:
# S Segment Tree with Collapsed Clades
options(repr.plot.width = 6.5, repr.plot.height = 15)

# Load ggforce for proper circular pie segments
library(ggforce)

root_node_S <- setdiff(S_tree@phylo$edge[, 1], S_tree@phylo$edge[, 2])

# Calculate number of tips for each node
node_descendants <- tibble(
    node = unique(c(S_tree_metadata@phylo$edge[, 1], S_tree_metadata@phylo$edge[, 2]))
) %>%
    mutate(n_tips = map_int(node, ~length(Descendants(S_tree@phylo, .x, type = "tips")[[1]])))

# Configuration for clade collapsing and labeling
clade_config <- tibble(
    node     = c(728, 618, 683, 565, 536, 657, 510, 502, 495, 482, 493, 667, 513, 708),
    label    = c("Faranah", "Kenema", "Faranah", "Kenema", "Kenema",
                 "Makeni", "Guéckédou", "Nzérékoré", "Gbargna", "Ganta", "Ganta", "Madina-Oula", "Gonakry", "Faranah"),
    collapse = c(TRUE, TRUE, FALSE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE,FALSE, TRUE),
    scale    = c(0.02, 0.05, 1, 0.05, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0.05),
    fill     = c("#E41A1C", "#377EB8", "#E41A1C", "#377EB8", "#377EB8",
                 "#377EB8", "#E41A1C", "#E41A1C", "#377EB8", "#377EB8", "#377EB8", "#E41A1C", "#E41A1C","#E41A1C")
) %>%
    left_join(node_descendants, by = "node")

# Separate collapsed and non-collapsed configs
collapsed_config <- filter(clade_config, collapse)
labeled_config   <- filter(clade_config, !collapse)

# Add clade info to tree metadata for nodelab
S_tree_metadata_clades <- S_tree_metadata
S_tree_metadata_clades@data <- S_tree_metadata_clades@data %>%
    left_join(
        clade_config %>% select(node, clade_label = label, clade_n_tips = n_tips, is_collapsed = collapse),
        by = "node"
    )

# Build base tree
S_tree_collapsed <- ggtree(S_tree_metadata_clades, mrsd = MRSD, ladderize = TRUE) +
    theme_tree2() +
    # Lineage labels
    geom_nodelab(
        aes(x = branch, subset = node == 500, label = "IVb"), nudge_y = 1.5, size = 5, fontface = "bold"
    ) +
    geom_nodelab(
        aes(x=branch, subset = node == 481, label = "IVa"), nudge_y = 1.5, size = 5, fontface = "bold"
    )

# Apply scaling to clades
plot_result <- reduce2(
    .x = clade_config$node,
    .y = clade_config$scale,
    .f = scaleClade,
    .init = S_tree_collapsed
)

# Apply collapsing to specified clades
plot_result <- reduce2(
    .x = collapsed_config$node,
    .y = collapsed_config$fill,
    .f = ~collapse(..1, node = ..2, mode = "max", fill = ..3, color = "black"),
    .init = plot_result
)

# Get tree data for pie chart preparation
tree_data <- plot_result$data

# Prepare the data for the pies - filter nodes with posterior > 0.8
pie_data <- tree_data %>%
  mutate(posterior_num = as.numeric(posterior)) %>%
  filter(!is.na(posterior_num)) %>%
  filter(node != root_node_S) %>%  # Use root_node_S instead of root_node_L
  # Filter out collapsed nodes
  filter(!(node %in% collapsed_config$node)) %>%
  filter(!isTip)

# Get the data ranges to calculate aspect ratio
x_range <- range(tree_data$x, na.rm = TRUE)
y_range <- range(tree_data$y, na.rm = TRUE)

# Plot dimensions
plot_width <- 8.5
plot_height <- 20

# Calculate the scaling factor needed for x coordinates to make circles appear round
x_units_per_inch <- diff(x_range) / plot_width
y_units_per_inch <- diff(y_range) / plot_height

# Aspect ratio: how much to stretch x relative to y
aspect_ratio <- x_units_per_inch / y_units_per_inch

#' Create pie polygon data that accounts for aspect ratio distortion
#' @param pie_data Data frame with x, y, node, and posterior_num columns
#' @param r_y Radius in y-axis units
#' @param aspect Aspect ratio correction factor (x_units/y_units per inch)
#' @param n_points Number of points to use for the circle
#' @return List of data frames for posterior slice, rest slice, and border
create_pie_polygons <- function(pie_data, r_y = 0.5, aspect = 1, n_points = 50, start_angle = pi/2) {
    lapply(1:nrow(pie_data), function(i) {
        row <- pie_data[i, ]
        post_val <- row$posterior_num

        # Create the posterior slice (from start_angle to start_angle + posterior * 2*pi)
        angles_post <- seq(start_angle, start_angle + post_val * 2 * pi,
                          length.out = max(3, round(n_points * post_val)))
        x_post <- row$x + r_y * aspect * cos(angles_post)
        y_post <- row$y + r_y * sin(angles_post)

        # Create the rest slice (from start_angle + posterior * 2*pi to start_angle + 2*pi)
        angles_rest <- seq(start_angle + post_val * 2 * pi, start_angle + 2 * pi,
                          length.out = max(3, round(n_points * (1 - post_val))))
        x_rest <- row$x + r_y * aspect * cos(angles_rest)
        y_rest <- row$y + r_y * sin(angles_rest)

        # Create border circle
        angles_circle <- seq(start_angle, start_angle + 2 * pi, length.out = n_points)
        x_circle <- row$x + r_y * aspect * cos(angles_circle)
        y_circle <- row$y + r_y * sin(angles_circle)

        list(
            posterior = data.frame(
                x = c(row$x, x_post, row$x),
                y = c(row$y, y_post, row$y),
                node = row$node,
                type = "posterior"
            ),
            rest = data.frame(
                x = c(row$x, x_rest, row$x),
                y = c(row$y, y_rest, row$y),
                node = row$node,
                type = "rest"
            ),
            border = data.frame(
                x = x_circle,
                y = y_circle,
                node = row$node,
                type = "border"
            )
        )
    })
}

# Create pie polygon data - adjust r_y to control pie size
PIE_RADIUS <- 0.55  # Adjust this value to change pie size

pie_polygons <- create_pie_polygons(pie_data, r_y = PIE_RADIUS, aspect = aspect_ratio)

# Combine into data frames
posterior_polys <- bind_rows(lapply(pie_polygons, function(p) p$posterior))
rest_polys <- bind_rows(lapply(pie_polygons, function(p) p$rest))
border_polys <- bind_rows(lapply(pie_polygons, function(p) p$border))

# Add labels for collapsed clades and pie charts
plot_result <- plot_result +
    # Tip count to the right of collapsed node
    geom_nodelab(
        aes(x = branch, subset = !is.na(clade_label) & is_collapsed, label = paste0("n=", clade_n_tips)),
        nudge_y = 1,
        size = 3
    ) +
    # Labels for non-collapsed clades (just city name on top)
    geom_nodelab(
        aes(x = branch, subset = !is.na(clade_label) & !is_collapsed, label = clade_label),
        nudge_y = 1,
        size = 3.5,
        fontface = "bold"
    ) +
    # Add the "rest" slice (light gray background)
    geom_polygon(
        data = rest_polys,
        aes(x = x, y = y, group = node),
        fill = "white",
        color = NA,
        inherit.aes = FALSE
    ) +
    # Add the posterior probability slice (filled dark gray)
    geom_polygon(
        data = posterior_polys,
        aes(x = x, y = y, group = node),
        fill = "gray40",
        color = NA,
        inherit.aes = FALSE
    ) +
    # Add border circles
    geom_path(
        data = border_polys,
        aes(x = x, y = y, group = node),
        color = "black",
        linewidth = 0.3,
        inherit.aes = FALSE
    ) +
    # Tip styling
    geom_tiplab(
        aes(
            subset = newsequence,
            label = label_from_seqid(
                ifelse(
                    str_ends(str_split(label, "\\|", simplify = TRUE)[, 2], "-S"),
                    str_split(label, "\\|", simplify = TRUE)[, 2],
                    paste0(str_split(label, "\\|", simplify = TRUE)[, 2], "-S")
                )
            )
        ), size = 3, offset = 1.5, align = TRUE
    ) +
    geom_tippoint(aes(fill = country), colour = "black", shape = 21, stroke = 1, size = 3) +
    scale_fill_manual(name = "Country", values = COUNTRY_COLORS, labels = c("Guinea", "Liberia","Sierra Leone"), na.value = "gray50",
        guide = guide_legend(override.aes = list(size = 5))) +
    theme(
        panel.grid.major.x = element_line(linetype = "dashed", color = "gray80", linewidth = 0.5),
        panel.grid.minor.x = element_line(linetype = "dotted", color = "gray90", linewidth = 0.3),
        legend.position = "inside",
        legend.position.inside = c(0.02, 0.99),
        legend.justification = c(0, 1),
        legend.background = element_rect(fill = "white", colour = "black", linewidth = 0.6),
        legend.margin = margin(8, 10, 8, 10),
        legend.title = element_text(size = 18, face = "bold"),
        legend.text = element_text(size = 16),
        legend.key.size = unit(1.2, "lines"),
        plot.margin = margin(10, 10, 10, 10)
    ) +
    scale_x_continuous(limits = c(1700,2050))

plot_result

### 8.2 L Segment Tree with Collapsed Clades

Visualization with specific clades collapsed and labeled.


In [ ]:
# L Segment Tree with Collapsed Clades
options(repr.plot.width = 6, repr.plot.height = 15)

# Load ggforce for proper circular pie segments
library(ggforce)

root_node_L <- setdiff(L_tree@phylo$edge[, 1], L_tree@phylo$edge[, 2])

# Calculate number of tips for each node
L_node_descendants <- tibble(
    node = unique(c(L_tree_metadata@phylo$edge[, 1], L_tree_metadata@phylo$edge[, 2]))
) %>%
    mutate(n_tips = map_int(node, ~length(Descendants(L_tree@phylo, .x, type = "tips")[[1]])))

# Configuration for clade collapsing and labeling
L_clade_config <- tibble(
    node     = c(289, 265, 235, 213, 199, 194, 179, 183, 161, 163, 230,168),
    label    = c("Kenema", "Kenema", "Kenema", "Makeni", "Guéckédou",
                 "Nzérékoré", "Faranah", "Conakry", "Ganta", "Gbarnga", "Madina-Oula", "Ganta"),
    collapse = c(TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE),
    scale    = c(0.05, 0.05, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1),
    fill     = c("#377EB8", "#377EB8", "#377EB8", "#377EB8", "#E41A1C",
                 "#E41A1C", "#E41A1C", "#E41A1C", "#377EB8", "#377EB8", "#E41A1C", "#E41A1C")
) %>%
    left_join(L_node_descendants, by = "node")



# FLI nodes (from comments)
FLIP_NODES <- tibble(
    x = 193,
    y = 179
)

# Separate collapsed and non-collapsed configs
L_collapsed_config <- filter(L_clade_config, collapse)
L_labeled_config   <- filter(L_clade_config, !collapse)

# Add clade info to tree metadata for nodelab
L_tree_metadata_clades <- L_tree_metadata
L_tree_metadata_clades@data <- L_tree_metadata_clades@data %>%
    left_join(
        L_clade_config %>% select(node, clade_label = label, clade_n_tips = n_tips, is_collapsed = collapse),
        by = "node"
    )

range_nodes <- L_tree_metadata_clades@data %>%
    filter(!is.na(clade_label) & !is_collapsed) %>%
    pull(node)

# Build base tree
L_tree_raw <- ggtree(L_tree_metadata_clades, mrsd = MRSD, ladderize = TRUE)

L_plot_flipped <-reduce2(
    .x = FLIP_NODES$x,
    .y = FLIP_NODES$y,
    .f = ~ggtree::flip(tree_view = ..1, ..2, ..3),
    .init = L_tree_raw
)

L_tree_collapsed <- L_plot_flipped +
    theme_tree2() +
    # Lineage labels
    geom_nodelab(
        aes(x=branch, subset = node == 178, label = "IVb"),
        nudge_y = 1, size = 5, fontface = "bold"
    ) +
    geom_nodelab(
        aes(x = branch, subset = node == root_node_L + 1, label = "IVa"),
        nudge_y = 1, size = 5, fontface = "bold"
    )


# Apply scaling to clades
L_plot_result <- reduce2(
    .x = L_clade_config$node,
    .y = L_clade_config$scale,
    .f = scaleClade,
    .init = L_tree_collapsed
)

# Apply collapsing to specified clades
L_plot_result <- reduce2(
    .x = L_collapsed_config$node,
    .y = L_collapsed_config$fill,
    .f = ~collapse(..1, node = ..2, mode = "max", fill = ..3, color = "black"),
    .init = L_plot_result
)

tree_data <- L_plot_result$data

# 2. Prepare the data for the pies - filter nodes with posterior > 0.8
pie_data <- tree_data %>%
  mutate(posterior_num = as.numeric(posterior)) %>%
  filter(!is.na(posterior_num)) %>%
  filter(node != root_node_L) %>%
  # L_clade_config%>%filter(collapse)%>%pull(node)
  filter(!(node %in% c(289, 265))) %>%
  filter(!isTip)

# 3. Create custom geom for pie charts that account for different x/y scales
# The key insight: we need to transform coordinates so the pie appears circular
# in the final plot despite different x/y scales

# Get the data ranges to calculate aspect ratio
x_range <- range(tree_data$x, na.rm = TRUE)
y_range <- range(tree_data$y, na.rm = TRUE)

# Plot dimensions
plot_width <- 8.5
plot_height <- 20

# Calculate the scaling factor needed for x coordinates to make circles appear round
# Data units per plot inch
x_units_per_inch <- diff(x_range) / plot_width
y_units_per_inch <- diff(y_range) / plot_height

# Aspect ratio: how much to stretch x relative to y
aspect_ratio <- x_units_per_inch / y_units_per_inch

#' Create pie polygon data that accounts for aspect ratio distortion
#' @param pie_data Data frame with x, y, node, and posterior_num columns
#' @param r_y Radius in y-axis units
#' @param aspect Aspect ratio correction factor (x_units/y_units per inch)
#' @param n_points Number of points to use for the circle
#' @return List of data frames for posterior slice, rest slice, and border
create_pie_polygons <- function(pie_data, r_y = 0.5, aspect = 1, n_points = 50, start_angle = pi/2) {
    lapply(1:nrow(pie_data), function(i) {
        row <- pie_data[i, ]
        post_val <- row$posterior_num

        # Create the posterior slice (from start_angle to start_angle + posterior * 2*pi)
        angles_post <- seq(start_angle, start_angle + post_val * 2 * pi,
                          length.out = max(3, round(n_points * post_val)))
        x_post <- row$x + r_y * aspect * cos(angles_post)
        y_post <- row$y + r_y * sin(angles_post)

        # Create the rest slice (from start_angle + posterior * 2*pi to start_angle + 2*pi)
        angles_rest <- seq(start_angle + post_val * 2 * pi, start_angle + 2 * pi,
                          length.out = max(3, round(n_points * (1 - post_val))))
        x_rest <- row$x + r_y * aspect * cos(angles_rest)
        y_rest <- row$y + r_y * sin(angles_rest)

        # Create border circle
        angles_circle <- seq(start_angle, start_angle + 2 * pi, length.out = n_points)
        x_circle <- row$x + r_y * aspect * cos(angles_circle)
        y_circle <- row$y + r_y * sin(angles_circle)

        list(
            posterior = data.frame(
                x = c(row$x, x_post, row$x),
                y = c(row$y, y_post, row$y),
                node = row$node,
                type = "posterior"
            ),
            rest = data.frame(
                x = c(row$x, x_rest, row$x),
                y = c(row$y, y_rest, row$y),
                node = row$node,
                type = "rest"
            ),
            border = data.frame(
                x = x_circle,
                y = y_circle,
                node = row$node,
                type = "border"
            )
        )
    })
}

# Create pie polygon data - adjust r_y to control pie size
# Larger r_y = larger pies
PIE_RADIUS <- 0.35  # Adjust this value to change pie size

pie_polygons <- create_pie_polygons(pie_data, r_y = PIE_RADIUS, aspect = aspect_ratio)

# Combine into data frames
posterior_polys <- bind_rows(lapply(pie_polygons, function(p) p$posterior))
rest_polys <- bind_rows(lapply(pie_polygons, function(p) p$rest))
border_polys <- bind_rows(lapply(pie_polygons, function(p) p$border))

# Add labels for collapsed clades and pie charts
L_plot_result <- L_plot_result +
    # Tip count to the right of collapsed node
    geom_nodelab(
        aes(x = branch, subset = !is.na(clade_label) & is_collapsed, label = paste0("n=", clade_n_tips)),
        nudge_y = 0.75,
        size = 3
    ) +
    # Labels for non-collapsed clades (just city name on top)
    geom_nodelab(
        aes(x = branch, subset = !is.na(clade_label) & !is_collapsed, label = clade_label),
        nudge_y = 0.75,
        size = 3.5,
        fontface = "bold"
    ) +
    geom_polygon(
        data = rest_polys,
        aes(x = x, y = y, group = node),
        fill = "white",
        color = NA,
        inherit.aes = FALSE
    ) +
    # Add the posterior probability slice (filled dark gray)
    geom_polygon(
        data = posterior_polys,
        aes(x = x, y = y, group = node),
        fill = "gray40",
        color = NA,
        inherit.aes = FALSE
    ) +
    # Add border circles
    geom_path(
        data = border_polys,
        aes(x = x, y = y, group = node),
        color = "black",
        linewidth = 0.3,
        inherit.aes = FALSE
    ) +
    # Tip styling
    geom_tippoint(aes(fill = country), colour = "black", shape = 21, stroke = 1, size = 3) +
    geom_tiplab(
        aes(
            subset = newsequence,
            label = label_from_seqid(
                ifelse(
                    str_ends(str_split(label, "\\|", simplify = TRUE)[, 2], "-L"),
                    str_split(label, "\\|", simplify = TRUE)[, 2],
                    paste0(str_split(label, "\\|", simplify = TRUE)[, 2], "-L")
                )
            )
        ), size = 3, offset = 0.7, align =TRUE
    ) +
    scale_fill_manual(name = "Country", values = COUNTRY_COLORS, labels = c("Guinea", "Liberia","Sierra Leone"), na.value = "gray50",
        guide = guide_legend(override.aes = list(size = 5))) +
    theme(
        panel.grid.major.x = element_line(linetype = "dashed", color = "gray80", linewidth = 0.5),
        panel.grid.minor.x = element_line(linetype = "dotted", color = "gray90", linewidth = 0.3),
        legend.position = "inside",
        legend.position.inside = c(0.02, 0.99),
        legend.justification = c(0, 1),
        legend.background = element_rect(fill = "white", colour = "black", linewidth = 0.6),
        legend.margin = margin(8, 10, 8, 10),
        legend.title = element_text(size = 18, face = "bold"),
        legend.text = element_text(size = 16),
        legend.key.size = unit(1.2, "lines"),
        plot.margin = margin(10, 10, 10, 10)
    )+
    scale_x_continuous(limits = c(1650,2050))


L_plot_result

In [ ]:
# L Segment Tree with Collapsed Clades
options(repr.plot.width = 6, repr.plot.height = 15)

# Load ggforce for proper circular pie segments
library(ggforce)

root_node_L <- setdiff(L_tree@phylo$edge[, 1], L_tree@phylo$edge[, 2])

# Calculate number of tips for each node
L_node_descendants <- tibble(
    node = unique(c(L_tree_metadata@phylo$edge[, 1], L_tree_metadata@phylo$edge[, 2]))
) %>%
    mutate(n_tips = map_int(node, ~length(Descendants(L_tree@phylo, .x, type = "tips")[[1]])))

# Configuration for clade collapsing and labeling
L_clade_config <- tibble(
    node     = c(289, 265, 235, 213, 199, 194, 179, 183, 161, 163, 230,168),
    label    = c("Kenema", "Kenema", "Kenema", "Makeni", "Guéckédou",
                 "Nzérékoré", "Faranah", "Conakry", "Ganta", "Gbarnga", "Madina-Oula", "Ganta"),
    collapse = c(TRUE, TRUE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE),
    scale    = c(0.05, 0.05, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1),
    fill     = c("#377EB8", "#377EB8", "#377EB8", "#377EB8", "#E41A1C",
                 "#E41A1C", "#E41A1C", "#E41A1C", "#377EB8", "#377EB8", "#E41A1C", "#E41A1C")
) %>%
    left_join(L_node_descendants, by = "node")



# FLI nodes (from comments)
FLIP_NODES <- tibble(
    x = 193,
    y = 179
)

# Separate collapsed and non-collapsed configs
L_collapsed_config <- filter(L_clade_config, collapse)
L_labeled_config   <- filter(L_clade_config, !collapse)

# Add clade info to tree metadata for nodelab
L_tree_metadata_clades <- L_tree_metadata
L_tree_metadata_clades@data <- L_tree_metadata_clades@data %>%
    left_join(
        L_clade_config %>% select(node, clade_label = label, clade_n_tips = n_tips, is_collapsed = collapse),
        by = "node"
    )

range_nodes <- L_tree_metadata_clades@data %>%
    filter(!is.na(clade_label) & !is_collapsed) %>%
    pull(node)

# Build base tree
L_tree_raw <- ggtree(L_tree_metadata_clades, mrsd = MRSD, ladderize = TRUE)

L_plot_flipped <-reduce2(
    .x = FLIP_NODES$x,
    .y = FLIP_NODES$y,
    .f = ~ggtree::flip(tree_view = ..1, ..2, ..3),
    .init = L_tree_raw
)

L_tree_collapsed <- L_plot_flipped +
    theme_tree2() +
    # Lineage labels
    geom_nodelab(
        aes(x=branch, subset = node == 178, label = "IVb"),
        nudge_y = 1, size = 5, fontface = "bold"
    ) +
    geom_nodelab(
        aes(x = branch, subset = node == root_node_L + 1, label = "IVa"),
        nudge_y = 1, size = 5, fontface = "bold"
    )


# Apply scaling to clades
L_plot_result <- reduce2(
    .x = L_clade_config$node,
    .y = L_clade_config$scale,
    .f = scaleClade,
    .init = L_tree_collapsed
)

# Apply collapsing to specified clades
L_plot_result <- reduce2(
    .x = L_collapsed_config$node,
    .y = L_collapsed_config$fill,
    .f = ~collapse(..1, node = ..2, mode = "max", fill = ..3, color = "black"),
    .init = L_plot_result
)

tree_data <- L_plot_result$data

# 2. Prepare the data for the pies - filter nodes with posterior > 0.8
pie_data <- tree_data %>%
  mutate(posterior_num = as.numeric(posterior)) %>%
  filter(!is.na(posterior_num)) %>%
  filter(node != root_node_L) %>%
  # L_clade_config%>%filter(collapse)%>%pull(node)
  filter(!(node %in% c(289, 265))) %>%
  filter(!isTip)

# 3. Create custom geom for pie charts that account for different x/y scales
# The key insight: we need to transform coordinates so the pie appears circular
# in the final plot despite different x/y scales

# Get the data ranges to calculate aspect ratio
x_range <- range(tree_data$x, na.rm = TRUE)
y_range <- range(tree_data$y, na.rm = TRUE)

# Plot dimensions
plot_width <- 8.5
plot_height <- 20

# Calculate the scaling factor needed for x coordinates to make circles appear round
# Data units per plot inch
x_units_per_inch <- diff(x_range) / plot_width
y_units_per_inch <- diff(y_range) / plot_height

# Aspect ratio: how much to stretch x relative to y
aspect_ratio <- x_units_per_inch / y_units_per_inch

#' Create pie polygon data that accounts for aspect ratio distortion
#' @param pie_data Data frame with x, y, node, and posterior_num columns
#' @param r_y Radius in y-axis units
#' @param aspect Aspect ratio correction factor (x_units/y_units per inch)
#' @param n_points Number of points to use for the circle
#' @return List of data frames for posterior slice, rest slice, and border
create_pie_polygons <- function(pie_data, r_y = 0.5, aspect = 1, n_points = 50, start_angle = pi/2) {
    lapply(1:nrow(pie_data), function(i) {
        row <- pie_data[i, ]
        post_val <- row$posterior_num

        # Create the posterior slice (from start_angle to start_angle + posterior * 2*pi)
        angles_post <- seq(start_angle, start_angle + post_val * 2 * pi,
                          length.out = max(3, round(n_points * post_val)))
        x_post <- row$x + r_y * aspect * cos(angles_post)
        y_post <- row$y + r_y * sin(angles_post)

        # Create the rest slice (from start_angle + posterior * 2*pi to start_angle + 2*pi)
        angles_rest <- seq(start_angle + post_val * 2 * pi, start_angle + 2 * pi,
                          length.out = max(3, round(n_points * (1 - post_val))))
        x_rest <- row$x + r_y * aspect * cos(angles_rest)
        y_rest <- row$y + r_y * sin(angles_rest)

        # Create border circle
        angles_circle <- seq(start_angle, start_angle + 2 * pi, length.out = n_points)
        x_circle <- row$x + r_y * aspect * cos(angles_circle)
        y_circle <- row$y + r_y * sin(angles_circle)

        list(
            posterior = data.frame(
                x = c(row$x, x_post, row$x),
                y = c(row$y, y_post, row$y),
                node = row$node,
                type = "posterior"
            ),
            rest = data.frame(
                x = c(row$x, x_rest, row$x),
                y = c(row$y, y_rest, row$y),
                node = row$node,
                type = "rest"
            ),
            border = data.frame(
                x = x_circle,
                y = y_circle,
                node = row$node,
                type = "border"
            )
        )
    })
}

# Create pie polygon data - adjust r_y to control pie size
# Larger r_y = larger pies
PIE_RADIUS <- 0.35  # Adjust this value to change pie size

pie_polygons <- create_pie_polygons(pie_data, r_y = PIE_RADIUS, aspect = aspect_ratio)

# Combine into data frames
posterior_polys <- bind_rows(lapply(pie_polygons, function(p) p$posterior))
rest_polys <- bind_rows(lapply(pie_polygons, function(p) p$rest))
border_polys <- bind_rows(lapply(pie_polygons, function(p) p$border))

# Add labels for collapsed clades and pie charts
L_plot_result <- L_plot_result +
    # Tip count to the right of collapsed node
    geom_nodelab(
        aes(x = branch, subset = !is.na(clade_label) & is_collapsed, label = paste0("n=", clade_n_tips)),
        nudge_y = 0.75,
        size = 3
    ) +
    # Labels for non-collapsed clades (just city name on top)
    geom_nodelab(
        aes(x = branch, subset = !is.na(clade_label) & !is_collapsed, label = clade_label),
        nudge_y = 0.75,
        size = 3.5,
        fontface = "bold"
    ) +
    geom_polygon(
        data = rest_polys,
        aes(x = x, y = y, group = node),
        fill = "white",
        color = NA,
        inherit.aes = FALSE
    ) +
    # Add the posterior probability slice (filled dark gray)
    geom_polygon(
        data = posterior_polys,
        aes(x = x, y = y, group = node),
        fill = "gray40",
        color = NA,
        inherit.aes = FALSE
    ) +
    # Add border circles
    geom_path(
        data = border_polys,
        aes(x = x, y = y, group = node),
        color = "black",
        linewidth = 0.3,
        inherit.aes = FALSE
    ) +
    # Tip styling
    geom_tippoint(aes(fill = country), colour = "black", shape = 21, stroke = 1, size = 3) +
    geom_tiplab(
        aes(
            subset = newsequence,
            label = label_from_seqid(
                ifelse(
                    str_ends(str_split(label, "\\|", simplify = TRUE)[, 2], "-L"),
                    str_split(label, "\\|", simplify = TRUE)[, 2],
                    paste0(str_split(label, "\\|", simplify = TRUE)[, 2], "-L")
                )
            )
        ), size = 3, offset = 0.7, align =TRUE
    ) +
    scale_fill_manual(name = "Country", values = COUNTRY_COLORS, labels = c("Guinea", "Liberia","Sierra Leone"), na.value = "gray50") +
    theme(
        panel.grid.major.x = element_line(linetype = "dashed", color = "gray80", linewidth = 0.5),
        panel.grid.minor.x = element_line(linetype = "dotted", color = "gray90", linewidth = 0.3),
        legend.position = "bottom",
        plot.margin = margin(10, 10, 10, 10)
    )+
    scale_x_continuous(limits = c(1650,2050))


L_plot_result

In [ ]:
LAYOUT <- '
AB
'

options(repr.plot.width = 13, repr.plot.height = 15)

tree_plots <- wrap_plots(
    A = L_plot_result,
    B = plot_result,
    design = LAYOUT,
    guides = "collect"
) +
    plot_annotation(
        tag_levels = list(c("A", "B"))
    ) &
    theme(legend.position = "bottom",
     plot.tag = element_text(size = 18, face = "bold"),
     legend.title = element_text(size = 14, face = "bold"),
     legend.text = element_text(size = 12),
     legend.key.size = unit(1, "cm"),
     axis.text.x = element_text(size = 12)
     )

tree_plots

ggsave("./figures/phylo_LS.pdf", tree_plots,
       width = 13, height = 15, device = cairo_pdf)